# Langchain Basics (QuickStart)

In [ ]:
# install the latest version of langchain
!pip install -U langchain

In [ ]:
# install the openai integration
!pip install -U langchain-openai

## Basic agent
Creating a simple agent that can answer questions and call tools. The agent will use openai as its language model, a basic weather function as a tool, and a simple prompt to guide its behavior.

In [21]:
# load api key and relevant libraries

import openai
import os


from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
llm_model = "openai/gpt-4.1"

In [7]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model=llm_model,api_key=token,base_url=endpoint)  # using openai integration throguh langchain-openai

In [11]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)
print(response["messages"][-1].content)

It's always sunny in San Francisco! If you need more detailed or current weather information, let me know.


## Build a real-world agent

In [13]:
# step 1 -  define system prompt with tool usage instructions
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

### Create tools

**Tools** let a model interact with external systems by calling functions you define. Tools can depend on **runtime context** and also interact with **agent memory**.
like how the get_user_location tool uses runtime context:

In [12]:
# step 2 - create tools 
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

In [27]:
# step 3 - configure model 
# set up the model using langchain's init_chat_model
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4.1",
    api_key = token,
    base_url = endpoint,
    temperature=0.5,
    timeout = 10,
    max_tokens=1000
)

In [29]:
# step 4 - Define response format(if want specific format)

from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [ ]:
#  step 5 - Add memory ( to maintain state across interactions)

from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [31]:
# Step 6 - Create and run the agent 
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

type(response['structured_response'])

ResponseFormat(punny_response="Florida's forecast is feeling bright—it's always sunny! No clouds to throw shade on your day, so let the good times roll like a gentle breeze.", weather_conditions="It's always sunny in Florida!")
ResponseFormat(punny_response='No prob-llama! If you ever need a weather pick-me-up, just give me a shout—I’ll make sure the forecast is never a bore-cast!', weather_conditions=None)


__main__.ResponseFormat